# FarmWise AI: Plant Disease Detection CNN (PlantVillage)
---
This notebook trains a **MobileNetV2 Convolutional Neural Network (CNN)** on the famous **PlantVillage Image Dataset**.

**Requirements:**
- Kaggle API Token (`kaggle.json` placed in `~/.kaggle/`)
- NVIDIA GPU (CUDA) deeply recommended. Training this on a CPU can take multipe hours.

This runs the download, structuring, training loop, and saves the `.h5` model to the backend folder.

In [ ]:
!pip install tensorflow pillow kaggle matplotlib

### 1. Download PlantVillage Dataset via Kaggle
Ensure your `kaggle.json` is set up.

In [ ]:
import os

# IMPORTANT: Run this if on Google Colab or replace with your local setup
# os.environ['KAGGLE_USERNAME'] = 'YOUR_USERNAME'
# os.environ['KAGGLE_KEY'] = 'YOUR_KEY'

if not os.path.exists('new-plant-diseases-dataset.zip'):
    print("Downloading PlantVillage Dataset... (800MB)")
    !kaggle datasets download -d vipoooool/new-plant-diseases-dataset
    !unzip -q new-plant-diseases-dataset.zip -d dataset
    print("Dataset Extracted.")
else:
    print("Dataset already exists.")

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

### 2. Data Generators and Augmentation

In [ ]:
TRAIN_DIR = 'dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train'
VAL_DIR = 'dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

val_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_data = val_gen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

NUM_CLASSES = len(train_data.class_indices)

### 3. Transfer Learning Architecture

In [ ]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
base_model.trainable = False  # Freeze for head-only initial training

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

### 4. Training Loop

In [ ]:
os.makedirs('../backend/app/models', exist_ok=True)
out_path = '../backend/app/models/disease_model.h5'

callbacks = [
    ModelCheckpoint(out_path, save_best_only=True, monitor='val_accuracy'),
    EarlyStopping(patience=4, restore_best_weights=True)
]

# Train just the head (10 epochs)
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks
)

In [ ]:
print(f"\u2705 Model saved to {out_path}")
print("Class Indices that must be mapped in FastAPI backend:")
print(train_data.class_indices)